In [1]:
%matplotlib inline

# plotting
import matplotlib as mpl
mpl.style.use('ggplot')
import matplotlib.pyplot as plt
import tensorflow

# math and data manipulation
import numpy as np
import pandas as pd

# to handle paths
from pathlib import Path

# set random seeds 
from numpy.random import seed
#from tensorflow import set_random_seed

# Exploratory Analysis

## Data1: Training data

In [51]:
df = pd.read_csv("cold-start-consumption-forecasting-training-data.csv", sep=";", parse_dates=['timestamp'])
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
df.sort_values(by='timestamp', inplace=True)

### training on only one building (site_id=2)

In [52]:
df2 = df[df["site_id"]==2]

In [53]:
df2

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
1221786,18469,2,2013-01-01 03:00:00+00:00,0.500000,1.000000,1.000000,52561.679251,51033.723459,49964.154405,53631.248306
29192,18470,2,2013-01-01 04:00:00+00:00,0.500000,0.500000,1.000000,55159.204098,52561.679251,51033.723459,62340.596321
56052,18471,2,2013-01-01 05:00:00+00:00,0.000000,0.500000,0.500000,58979.093579,55159.204098,52561.679251,106345.723136
1332108,18472,2,2013-01-01 06:00:00+00:00,0.000000,0.000000,0.500000,58520.706841,58979.093579,55159.204098,132473.767183
1063812,18473,2,2013-01-01 07:00:00+00:00,0.000000,0.000000,0.000000,59590.275895,58520.706841,58979.093579,160129.767021
...,...,...,...,...,...,...,...,...,...,...
438680,32888,2,2014-08-24 22:00:00+00:00,18.000000,18.500000,20.166667,65396.507906,66771.668119,65854.894643,173117.391255
438681,32889,2,2014-08-24 23:00:00+00:00,17.933333,18.000000,18.500000,66313.281381,65396.507906,66771.668119,177854.054211
359458,32890,2,2014-08-25 00:00:00+00:00,18.000000,17.933333,18.000000,72730.695708,66313.281381,65396.507906,172200.617780
1054107,32891,2,2014-08-25 01:00:00+00:00,17.000000,18.000000,17.933333,77314.563085,72730.695708,66313.281381,170978.253146


In [34]:
train2 = df2[:-24]
valid2 = df2[-24:]

In [35]:
valid2

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
698361,32869,2,2014-08-24 03:00:00+00:00,15.000000,15.800000,16.000000,62646.187480,62798.983059,63410.165376,73036.286867
438678,32870,2,2014-08-24 04:00:00+00:00,15.000000,15.000000,15.800000,62035.005163,62646.187480,62798.983059,75481.016134
1054105,32871,2,2014-08-24 05:00:00+00:00,14.166667,15.000000,15.000000,62340.596321,62035.005163,62646.187480,167769.545982
448850,32872,2,2014-08-24 06:00:00+00:00,14.000000,14.166667,15.000000,60659.844950,62340.596321,62035.005163,189160.927073
698362,32873,2,2014-08-24 07:00:00+00:00,15.500000,14.000000,14.166667,58062.320103,60659.844950,62340.596321,244472.926750
1226299,32874,2,2014-08-24 08:00:00+00:00,16.633333,15.500000,14.000000,58215.115682,58062.320103,60659.844950,249820.772023
1226301,32875,2,2014-08-24 09:00:00+00:00,19.000000,16.633333,15.500000,58367.911262,58215.115682,58062.320103,255015.821716
913111,32876,2,2014-08-24 10:00:00+00:00,20.000000,19.000000,16.633333,58673.502420,58367.911262,58215.115682,259141.302355
805669,32877,2,2014-08-24 11:00:00+00:00,20.433333,20.000000,19.000000,60048.662633,58673.502420,58367.911262,245848.086963
359454,32878,2,2014-08-24 12:00:00+00:00,21.500000,20.433333,20.000000,59743.071475,60048.662633,58673.502420,249667.976444


In [55]:
from autots import AutoTS
from sklearn.model_selection import train_test_split

In [56]:
# Define your model
model = AutoTS(
    forecast_length=24,  # adjust based on how far you want to forecast
    frequency='H',  # 'infer' will automatically detect the frequency
    ensemble='simple',  # use a simple ensemble of models
    model_list="fast_parallel",  # "superfast", "default", "fast_parallel"
    transformer_list="fast",  # "superfast",
    max_generations=1,  # number of iterations for model search
    num_validations=1,  # number of validations
    validation_method="backwards",  # type of validation
)

# Fit the model
model = model.fit(
    train2,
    date_col='timestamp',  # column with dates
    value_col='target',    # column with values to predict
    id_col='site_id',      # grouping (if applicable)
)

# Get the best model and predictions
prediction = model.predict()
print(prediction)

# Evaluate the model
forecast = prediction.forecast  # DataFrame of the forecast
print(forecast)
print("finished!")

Data frequency is: H, used frequency is: H
Model Number: 1 with model ARIMA in generation 0 of 1


 This problem is unconstrained.


RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =           17     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  1.09344D+01    |proj g|=  4.25758D-02

At iterate    1    f=  1.09306D+01    |proj g|=  2.86714D-02

At iterate    2    f=  1.09292D+01    |proj g|=  1.64552D-02

At iterate    3    f=  1.09277D+01    |proj g|=  1.19781D-02

At iterate    4    f=  1.09269D+01    |proj g|=  1.13415D-02

At iterate    5    f=  1.09267D+01    |proj g|=  1.03303D-02

At iterate    6    f=  1.09266D+01    |proj g|=  4.82664D-03

At iterate    7    f=  1.09265D+01    |proj g|=  5.76592D-03

At iterate    8    f=  1.09264D+01    |proj g|=  6.88976D-03

At iterate    9    f=  1.09262D+01    |proj g|=  7.80060D-03

At iterate   10    f=  1.09261D+01    |proj g|=  8.34055D-03

At iterate   11    f=  1.09260D+01    |proj g|=  5.16205D-03

At iterate   12    f=  1.09258D+01    |proj g|=  5.21981D-03

At iterate   13    f=  1.0

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/svm/_base.py:1250: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Model Number: 7 with model DatepartRegression in generation 0 of 1


...

31 - KalmanStateSpace with avg smape 57.75: 
Model Number: 32 of 53 with model DatepartRegression for Validation 1
Epoch 1/50
449/449 [==============================] - 2s 1ms/step - loss: 0.3216
Epoch 2/50
449/449 [==============================] - 0s 1ms/step - loss: 0.2855
Epoch 3/50
449/449 [==============================] - 0s 1ms/step - loss: 0.2708
Epoch 4/50
449/449 [==============================] - 0s 1ms/step - loss: 0.2354
Epoch 5/50
449/449 [==============================] - 0s 1ms/step - loss: 0.1903
Epoch 6/50
449/449 [==============================] - 0s 1ms/step - loss: 0.1565
Epoch 7/50
449/449 [==============================] - 0s 1ms/step - loss: 0.1444
Epoch 8/50
449/449 [==============================] - 0s 1ms/step - loss: 0.1352
Epoch 9/50
449/449 [==============================] - 0s 1ms/step - loss: 0.1292
Epoch 10/50
449/449 [==============================] - 0s 1ms/step - loss: 0.1248
Epoch 11/50
449/449 [==============================] - 0s 1ms/step - loss:

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/neural_network/_multilayer_perceptron.py:546: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


36 - MetricMotif with avg smape 9.61: 
Model Number: 37 of 53 with model Cassandra for Validation 1


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/autots/models/cassandra.py:1177: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_adjust.iloc[0] = 0.5


37 - Cassandra with avg smape 37.66: 
Model Number: 38 of 53 with model SeasonalNaive for Validation 1
38 - SeasonalNaive with avg smape 42.08: 
Model Number: 39 of 53 with model SeasonalNaive for Validation 1
39 - SeasonalNaive with avg smape 43.76: 
Model Number: 40 of 53 with model WindowRegression for Validation 1
40 - WindowRegression with avg smape 45.47: 
Model Number: 41 of 53 with model UnobservedComponents for Validation 1
41 - UnobservedComponents with avg smape 23.59: 
Model Number: 42 of 53 with model UnobservedComponents for Validation 1
42 - UnobservedComponents with avg smape 43.41: 
Model Number: 43 of 53 with model SectionalMotif for Validation 1
43 - SectionalMotif with avg smape 54.14: 
Model Number: 44 of 53 with model UnobservedComponents for Validation 1
44 - UnobservedComponents with avg smape 37.67: 
Model Number: 45 of 53 with model ARDL for Validation 1
45 - ARDL with avg smape 8.98: 
Model Number: 46 of 53 with model FFT for Validation 1
46 - FFT with avg sm

In [57]:
print(model.best_model_name)
print(model.best_model_params)
print(model.best_model_transformation_params)

Ensemble
{'model_name': 'BestN', 'model_count': 3, 'model_metric': 'bestn_horizontal', 'models': {'17db5ee619a255b26be9fb18e03c43c2': {'Model': 'SeasonalityMotif', 'ModelParameters': '{"window": 5, "point_method": "weighted_mean", "distance_metric": "mse", "k": 10, "datepart_method": "expanded"}', 'TransformationParameters': '{"fillna": "ffill", "transformations": {"0": "AlignLastValue", "1": "AlignLastValue", "2": "AlignLastValue", "3": "ClipOutliers"}, "transformation_params": {"0": {"rows": 1, "lag": 1, "method": "additive", "strength": 1.0, "first_value_only": false}, "1": {"rows": 1, "lag": 1, "method": "additive", "strength": 1.0, "first_value_only": false}, "2": {"rows": 7, "lag": 1, "method": "multiplicative", "strength": 1.0, "first_value_only": false}, "3": {"method": "clip", "std_threshold": 3.5, "fillna": null}}}'}, 'a84c237628610389761aef3fe9b4928f': {'Model': 'BallTreeMultivariateMotif', 'ModelParameters': '{"window": 10, "point_method": "median", "distance_metric": "mink

## Data2: Test data

In [4]:
data2 = pd.read_csv("cold-start-consumption-forecasting-test-data.csv", sep=";", parse_dates=['timestamp'])
data2

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
0,2,78,2017-03-06 15:00:00-05:00,8.000000,8.933333,10.500000,5.245016e+05,5.504866e+05,6.558341e+05,5.133429e+05
1,13,78,2017-07-13 10:00:00-04:00,31.000000,30.000000,29.000000,1.262663e+06,1.245585e+06,1.265211e+06,1.191620e+06
2,16,78,2017-04-05 14:00:00-04:00,16.000000,16.800000,16.500000,6.293816e+05,7.560991e+05,9.208202e+05,7.175366e+05
3,24,78,2016-12-05 01:00:00-05:00,4.500000,6.000000,6.133333,6.502152e+05,5.204650e+05,3.412245e+05,6.782952e+05
4,30,78,2017-09-12 18:00:00-04:00,16.000000,16.000000,17.533333,6.814274e+05,7.720909e+05,8.267658e+05,6.585562e+05
...,...,...,...,...,...,...,...,...,...,...
5371,5326,91,2015-07-28 19:00:00-04:00,23.466667,24.220000,25.420000,1.924487e+03,1.893677e+03,1.936338e+03,0.000000e+00
5372,5353,91,2016-08-13 21:00:00-04:00,17.600000,18.783333,20.000000,1.476546e+03,1.491952e+03,1.520392e+03,0.000000e+00
5373,5354,91,2016-05-24 15:00:00-04:00,19.833333,21.333333,21.533333,1.540538e+03,1.750288e+03,2.842885e+03,0.000000e+00
5374,5360,91,2015-06-20 01:00:00-04:00,19.450000,19.516667,19.866667,1.944633e+03,1.947003e+03,1.976629e+03,0.000000e+00


In [5]:
data2.describe()

,obs_id,site_id,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
count,5376.000000,5376.000000,5376.000000,5376.000000,5376.000000,5.376000e+03,5.376000e+03,5.376000e+03,5.376000e+03
mean,2687.500000,84.500000,15.202782,15.191096,15.179296,8.154911e+04,8.183575e+04,8.205588e+04,1.023380e+04
std,1552.061854,4.031504,8.432832,8.421938,8.396866,1.932085e+05,1.941248e+05,1.949242e+05,7.220886e+04
min,0.000000,78.000000,-9.645455,-10.800000,-11.408333,3.644937e+01,3.617219e+01,3.617219e+01,0.000000e+00
25%,1343.750000,81.000000,8.531250,8.500000,8.729096,3.847368e+03,3.935141e+03,3.945115e+03,0.000000e+00
50%,2687.500000,84.500000,15.000000,15.025000,15.073864,1.397601e+04,1.377829e+04,1.378522e+04,0.000000e+00
75%,4031.250000,88.000000,21.500000,21.533333,21.566667,4.210523e+04,4.243047e+04,4.231121e+04,0.000000e+00
max,5375.000000,91.000000,37.500000,38.100000,36.000000,1.343137e+06,1.306128e+06,1.337277e+06,1.231521e+06


In [6]:
validation = data2[data2["target"] != 0]
validation

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
0,2,78,2017-03-06 15:00:00-05:00,8.000000,8.933333,10.500000,5.245016e+05,5.504866e+05,6.558341e+05,5.133429e+05
1,13,78,2017-07-13 10:00:00-04:00,31.000000,30.000000,29.000000,1.262663e+06,1.245585e+06,1.265211e+06,1.191620e+06
2,16,78,2017-04-05 14:00:00-04:00,16.000000,16.800000,16.500000,6.293816e+05,7.560991e+05,9.208202e+05,7.175366e+05
3,24,78,2016-12-05 01:00:00-05:00,4.500000,6.000000,6.133333,6.502152e+05,5.204650e+05,3.412245e+05,6.782952e+05
4,30,78,2017-09-12 18:00:00-04:00,16.000000,16.000000,17.533333,6.814274e+05,7.720909e+05,8.267658e+05,6.585562e+05
...,...,...,...,...,...,...,...,...,...,...
5330,4621,90,2015-12-29 23:00:00-05:00,1.233333,2.000000,3.000000,1.337628e+04,1.883139e+04,2.256224e+04,2.057903e+04
5331,4644,90,2015-09-04 08:00:00-04:00,22.500000,21.066667,20.000000,9.678508e+04,3.182673e+04,1.775206e+04,7.645166e+04
5355,4996,91,2015-12-12 11:00:00-05:00,10.550000,11.733333,11.416667,4.424899e+03,3.588268e+03,2.794299e+03,3.058560e+03
5356,5009,91,2015-06-20 07:00:00-04:00,24.850000,23.216667,21.883333,2.232595e+03,2.102242e+03,1.996774e+03,2.306067e+03


In [7]:
test = data2[data2["target"] == 0]
test

,obs_id,site_id,timestamp,temperature,temperature_-1,temperature_-2,load,load_-1,load_-2,target
5,51,78,2017-05-15 03:00:00-04:00,15.500000,13.000000,12.900000,1.006653e+06,748475.882297,5.781718e+05,0.0
6,55,78,2016-12-15 10:00:00-05:00,2.000000,2.000000,1.833333,8.733329e+05,941881.203449,1.029805e+06,0.0
7,60,78,2017-02-03 11:00:00-05:00,8.133333,9.000000,9.000000,5.052074e+05,590755.741385,5.791804e+05,0.0
8,70,78,2017-03-17 14:00:00-04:00,18.766667,19.700000,20.550000,6.957117e+05,848383.949112,9.185337e+05,0.0
9,77,78,2017-01-06 12:00:00-05:00,0.000000,1.733333,2.500000,2.062510e+05,172491.350176,2.211416e+05,0.0
...,...,...,...,...,...,...,...,...,...,...
5371,5326,91,2015-07-28 19:00:00-04:00,23.466667,24.220000,25.420000,1.924487e+03,1893.676550,1.936338e+03,0.0
5372,5353,91,2016-08-13 21:00:00-04:00,17.600000,18.783333,20.000000,1.476546e+03,1491.951675,1.520392e+03,0.0
5373,5354,91,2016-05-24 15:00:00-04:00,19.833333,21.333333,21.533333,1.540538e+03,1750.288025,2.842885e+03,0.0
5374,5360,91,2015-06-20 01:00:00-04:00,19.450000,19.516667,19.866667,1.944633e+03,1947.002861,1.976629e+03,0.0


## Data3: Submission data

In [8]:
submission = pd.read_csv("cold-start-consumption-forecasting-submission-data.csv", sep=";", parse_dates=['timestamp'])
submission

,obs_id,site_id,timestamp,leaderboard_target,target
0,3013,86,2017-05-03 20:00:00-04:00,private,0.0
1,3018,86,2017-03-13 02:00:00-04:00,private,0.0
2,3062,87,2015-03-20 16:00:00-04:00,public,0.0
3,3081,87,2015-07-30 02:00:00-04:00,public,0.0
4,3090,87,2016-09-30 21:00:00-04:00,public,0.0
...,...,...,...,...,...
4699,4666,91,2015-09-05 08:00:00-04:00,private,0.0
4700,4670,91,2015-07-16 22:00:00-04:00,private,0.0
4701,4679,91,2016-03-15 19:00:00-04:00,private,0.0
4702,4686,91,2015-09-18 11:00:00-04:00,private,0.0


In [9]:
submission.describe()

,obs_id,site_id,target
count,4704.000000,4704.000000,4704.0
mean,2351.500000,84.500000,0.0
std,1358.072163,4.031557,0.0
min,0.000000,78.000000,0.0
25%,1175.750000,81.000000,0.0
50%,2351.500000,84.500000,0.0
75%,3527.250000,88.000000,0.0
max,4703.000000,91.000000,0.0


## Data4: Meta data

In [10]:
metadata = pd.read_csv("cold-start-consumption-forecasting-meta-data.csv", sep=";")
metadata

,obs_id,site_id,surface,base_temperature,monday_is_day_off,tuesday_is_day_off,wednesday_is_day_off,thursday_is_day_off,friday_is_day_off,saturday_is_day_off,sunday_is_day_off
0,1,88,17.321667,18.0,False,False,False,False,False,True,True
1,39,65,700.616167,18.0,False,False,False,False,False,True,True
2,77,46,487.124678,18.0,False,False,False,False,False,True,True
3,0,28,1334.976110,18.0,False,False,False,False,False,True,True
4,12,2,10060.060936,18.0,False,False,False,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...
86,38,14,16355.878746,18.0,False,False,False,False,False,True,True
87,55,53,5996.254722,18.0,False,False,False,False,False,True,True
88,70,85,1887.420261,18.0,False,False,False,False,False,True,True
89,71,50,13357.690454,18.0,False,False,False,False,False,True,True


In [11]:
metadata.describe()

,obs_id,site_id,surface,base_temperature
count,91.00000,91.00000,91.000000,91.000000
mean,45.00000,46.00000,8201.376102,18.021978
std,26.41338,26.41338,11379.584643,0.209657
min,0.00000,1.00000,14.884534,18.000000
25%,22.50000,23.50000,1118.799075,18.000000
50%,45.00000,46.00000,3580.148991,18.000000
75%,67.50000,68.50000,12281.567083,18.000000
max,90.00000,91.00000,71615.852476,20.000000


## Data5: Holidays data

In [12]:
holidays = pd.read_csv("cold-start-consumption-forecasting-holidays-data.csv", sep=";")
holidays

,obs_id,site_id,holiday,date
0,20,20,Confederate Heroes Day,2017-01-19
1,32,20,Texas Independence Day (Observed),2014-03-03
2,47,20,New year,2015-01-01
3,112,34,Memorial Day,2015-05-25
4,122,34,"Birthday of Martin Luther King, Jr.",2017-01-16
...,...,...,...,...
3282,3211,53,National Holiday,2017-10-26
3283,3216,53,All Saints Day,2016-11-01
3284,3232,56,New year,2016-01-01
3285,3239,56,Assumption of Mary to Heaven,2015-08-15


In [13]:
holidays.describe()

,obs_id,site_id
count,3287.000000,3287.000000
mean,1643.000000,45.721935
std,949.019494,26.162549
min,0.000000,1.000000
25%,821.500000,22.000000
50%,1643.000000,46.000000
75%,2464.500000,68.000000
max,3286.000000,91.000000
